In [1]:
import pulp
import numpy as np
import pandas as pd

In [2]:
# Define constants
#

DELTA_T = 0.25  # 15 min in Hours
N = 96  # Number of horizon time steps

BAT_CAP = 5.12*4                # kWh
BAT_P_MAX = 4.0*3               # kW
ETA_CH = np.sqrt(0.95) * 0.96   # 95% BESS round-trip efficiency * 96% inverter power factor
ETA_DIS = np.sqrt(0.95) * 0.96  # 95% BESS round-trip efficiency * 96% inverter power factor
SOC_MIN = 0.1 * BAT_CAP
SOC_MAX = 0.9 * BAT_CAP


In [3]:
import pulp
import numpy as np
import pandas as pd


def optimize_day(
    prices_epex: pd.Series,   # ct/kWh, Länge 96
    pv: pd.Series,            # kW
    load: pd.Series,          # kW
    soc0: float,              # kWh
) -> dict:

    # --- Konsistenzcheck ---
    assert prices_epex.index.equals(pv.index)
    assert pv.index.equals(load.index)

    time_index = prices_epex.index
    N = len(time_index)
    T = range(N)

    # Preise in ct/kWh (laut vkw.at am 22.12.2025)
    price_sell = prices_epex - 0.6
    price_buy  = prices_epex + 1.44

    # Gewinnmaximierungsproblem
    model = pulp.LpProblem("DayAheadOpt", pulp.LpMaximize)

    # Variablen
    p_ch   = pulp.LpVariable.dicts("p_ch", T, 0, BAT_P_MAX)
    p_dis  = pulp.LpVariable.dicts("p_dis", T, 0, BAT_P_MAX)
    soc    = pulp.LpVariable.dicts("soc", T, SOC_MIN, SOC_MAX)

    p_sell = pulp.LpVariable.dicts("p_sell", T, 0)
    p_buy  = pulp.LpVariable.dicts("p_buy", T, 0)

    # Binärvariable gegen gleichzeitiges Laden/Entladen
    y = pulp.LpVariable.dicts("y", T, 0, 1, cat="Binary")

    # SOC-Dynamik
    for t in T:
        if t == 0:
            model += soc[t] == soc0 + DELTA_T * (
                ETA_CH * p_ch[t] - p_dis[t] / ETA_DIS
            )
        else:
            model += soc[t] == soc[t-1] + DELTA_T * (
                ETA_CH * p_ch[t] - p_dis[t] / ETA_DIS
            )

    # Lade-/Entlade-Exklusivität
    for t in T:
        model += p_ch[t]  <= BAT_P_MAX * y[t]
        model += p_dis[t] <= BAT_P_MAX * (1 - y[t])

    # Leistungsbilanz
    for t in T:
        model += (
            pv.iloc[t] + p_buy[t] + p_dis[t]
            ==
            load.iloc[t] + p_sell[t] + p_ch[t]
        )

    # Zielfunktion: Erlös – Kosten
    model += pulp.lpSum(
        (price_sell.iloc[t] * p_sell[t]
         - price_buy.iloc[t] * p_buy[t]) * DELTA_T
        for t in T
    )

    model.solve(pulp.PULP_CBC_CMD(msg=False))

    # --- Rückgabe als pandas.Series ---
    return {
        "soc":    pd.Series([soc[t].value() for t in T], index=time_index),
        "p_ch":   pd.Series([p_ch[t].value() for t in T], index=time_index),
        "p_dis":  pd.Series([p_dis[t].value() for t in T], index=time_index),
        "p_sell": pd.Series([p_sell[t].value() for t in T], index=time_index),
        "p_buy":  pd.Series([p_buy[t].value() for t in T], index=time_index),
    }


In [20]:
act_timerange = pd.date_range(
    start="2025-12-18 00:00",
    end="2025-12-18 23:45",
    freq="15min",
    tz="Europe/Berlin",
    )

df_energy = pd.read_csv(
    "../data/energy_data.csv",
    index_col=0,
    parse_dates=True,
    )
pv_forecast = df_energy["Production"].loc[act_timerange]
load_forecast = df_energy["Consumption"].loc[act_timerange]

df_prices_epex = pd.read_csv(
    "../data/day_ahead_prices.csv",
    index_col=0,
    parse_dates=True,
    )
prices_epex = df_prices_epex["day_ahead_price_EUR_MWh"].loc[act_timerange]

result = optimize_day(
    prices_epex=prices_epex,
    pv=pv_forecast,
    load=load_forecast,
    soc0=0.5 * BAT_CAP,
)

result["soc"].plot()
result["p_sell"].plot()


KeyError: "None of [DatetimeIndex(['2025-12-18 00:00:00+01:00', '2025-12-18 00:15:00+01:00',\n               '2025-12-18 00:30:00+01:00', '2025-12-18 00:45:00+01:00',\n               '2025-12-18 01:00:00+01:00', '2025-12-18 01:15:00+01:00',\n               '2025-12-18 01:30:00+01:00', '2025-12-18 01:45:00+01:00',\n               '2025-12-18 02:00:00+01:00', '2025-12-18 02:15:00+01:00',\n               '2025-12-18 02:30:00+01:00', '2025-12-18 02:45:00+01:00',\n               '2025-12-18 03:00:00+01:00', '2025-12-18 03:15:00+01:00',\n               '2025-12-18 03:30:00+01:00', '2025-12-18 03:45:00+01:00',\n               '2025-12-18 04:00:00+01:00', '2025-12-18 04:15:00+01:00',\n               '2025-12-18 04:30:00+01:00', '2025-12-18 04:45:00+01:00',\n               '2025-12-18 05:00:00+01:00', '2025-12-18 05:15:00+01:00',\n               '2025-12-18 05:30:00+01:00', '2025-12-18 05:45:00+01:00',\n               '2025-12-18 06:00:00+01:00', '2025-12-18 06:15:00+01:00',\n               '20

In [ ]:
type(prices_epex)


pandas.core.series.Series

In [ ]:
SOC = 10.0  # Start-SOC

for k in range(N_sim_steps):

    prices_h = prices[k : k+H]
    pv_h     = pv_forecast(k, H)
    load_h   = load_forecast(k, H)

    model = build_model(prices_h, pv_h, load_h, SOC)
    solver.solve(model)

    # Nur Schritt 0 umsetzen
    p_ch  = pyo.value(model.p_ch[0])
    p_dis = pyo.value(model.p_dis[0])

    # System simulieren
    SOC = SOC + DELTA_T * (ETA_CH * p_ch - p_dis / ETA_DIS)

    log_step(k, SOC, p_ch, p_dis)


NameError: name 'N_sim_steps' is not defined